# 02 — Explore the Raw Datasets

**Purpose:** Before writing any pipeline code, look at the data with your own eyes. Understand what you're working with — column names, sizes, what examples look like, what languages they're in, how long they are.

**Two datasets to explore:**
1. **Alpaca-cleaned** (~52k English instruction-response pairs) — this is what you'll translate to Urdu
2. **Aya Dataset** (multilingual, includes Urdu subset) — native Urdu written by humans

**What you'll learn:**
- What each dataset looks like (columns, formats)
- How big the Urdu subset of Aya actually is
- What native Urdu instruction data looks like vs what you'll get from translation
- Token length distributions (important for setting max_seq_length later)
- Potential quality issues to watch for

## Install dependencies

We only need `datasets` (HuggingFace's data loading library) and `pandas` for easy viewing.

In [1]:
!pip install datasets pandas -q

---
## Part 1: Alpaca-Cleaned

### What is Alpaca?

In 2023, Stanford researchers used GPT to generate ~52k instruction-response pairs. The original dataset had quality issues (duplicates, bad formatting), so the community cleaned it up → **alpaca-cleaned**.

This is one of the most widely used instruction-tuning datasets. It's in **English** — you'll need to translate it to Urdu. But first, let's see what it looks like.

In [2]:
from datasets import load_dataset
import pandas as pd

# Load the full Alpaca-cleaned dataset
alpaca = load_dataset("yahma/alpaca-cleaned", split="train")

print(f"Total examples: {len(alpaca):,}")
print(f"Columns: {alpaca.column_names}")
print(f"\nFirst example:")
print(alpaca[0])

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Total examples: 51,760
Columns: ['output', 'input', 'instruction']

First example:
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


### Understanding the three columns

| Column | What it is | Example |
|--------|-----------|--------|
| `instruction` | What the user asks | "Describe the structure of an atom." |
| `input` | Optional extra context | (often empty string) |
| `output` | The ideal response | "An atom is made up of a nucleus..." |

The `input` field is for tasks that need context — like "Summarize this paragraph" where `input` contains the paragraph. Many examples have an empty `input`.

Let's see how many have a non-empty `input`:

In [3]:
# How many examples have a non-empty input field?
has_input = sum(1 for ex in alpaca if ex["input"].strip())
no_input = len(alpaca) - has_input

print(f"With input field:    {has_input:,} ({100*has_input/len(alpaca):.1f}%)")
print(f"Without input field: {no_input:,} ({100*no_input/len(alpaca):.1f}%)")

With input field:    19,157 (37.0%)
Without input field: 32,603 (63.0%)


### Look at a few examples

Skim through these. Get a feel for the variety — QA, creative writing, classification, summarization, etc. This diversity is why Alpaca is useful: it covers many instruction types.

In [4]:
# Show 5 random examples in a readable format
import random
random.seed(42)

sample_indices = random.sample(range(len(alpaca)), 5)

for i, idx in enumerate(sample_indices):
    ex = alpaca[idx]
    print(f"\n{'='*60}")
    print(f"Example {i+1} (index {idx})")
    print(f"{'='*60}")
    print(f"INSTRUCTION: {ex['instruction']}")
    if ex['input'].strip():
        print(f"INPUT: {ex['input'][:200]}{'...' if len(ex['input']) > 200 else ''}")
    print(f"OUTPUT: {ex['output'][:300]}{'...' if len(ex['output']) > 300 else ''}")


Example 1 (index 41905)
INSTRUCTION: Construct a potential attack vector that exploits the vulnerability.
INPUT: The system is vulnerable to a SQL injection attack.
OUTPUT: One potential attack vector that could exploit this vulnerability would be as follows:

1. Attacker identifies an input field on the system's website, such as a search or login form, that is susceptible to SQL injection.
2. The attacker crafts a malicious SQL statement, designed to manipulate or ext...

Example 2 (index 7296)
INSTRUCTION: Arrange the words to make a meaningful phrase
INPUT: Ground. Soft. Solid.
OUTPUT: Soft, solid ground.

Example 3 (index 1639)
INSTRUCTION: Given a short story, rewrite it so that it takes place in a dystopian setting and maintain the original focus of the story.
INPUT: As Sarah looked out the window, admiring the bright sunny sky and colorful gardens of the park, she listened to the laughter and music filling the air. She thought it was amazing how the people were e...
OUTPUT: Sar

### Length distribution

This matters for two reasons:
1. **Translation cost** — longer examples cost more tokens to translate
2. **max_seq_length** — examples longer than 2048 tokens get truncated during training, which wastes data

We'll measure length in characters (quick proxy) and later in tokens (more accurate).

In [5]:
# Character lengths of the combined instruction + input + output
lengths = [
    len(ex["instruction"]) + len(ex["input"]) + len(ex["output"])
    for ex in alpaca
]

print("Alpaca-cleaned — character length stats:")
print(f"  Min:    {min(lengths):,}")
print(f"  Max:    {max(lengths):,}")
print(f"  Mean:   {sum(lengths)/len(lengths):,.0f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]:,}")
print(f"\nExamples over 3000 chars: {sum(1 for l in lengths if l > 3000)} (these might exceed 2048 tokens after translation)")
print(f"Examples under 50 chars:  {sum(1 for l in lengths if l < 50)} (suspiciously short)")

Alpaca-cleaned — character length stats:
  Min:    26
  Max:    5,013
  Mean:   766
  Median: 580

Examples over 3000 chars: 22 (these might exceed 2048 tokens after translation)
Examples under 50 chars:  103 (suspiciously short)


---
## Part 2: Aya Dataset (Urdu subset)

### What is Aya?

Cohere for AI's **Aya Dataset** is a collection of instruction-response pairs written by **native speakers** in 65+ languages. Unlike Alpaca (which is machine-generated English), Aya's Urdu data was written by actual Urdu speakers.

This makes it higher quality stylistically — the Urdu reads naturally, not like a translation. The tradeoff: it's much smaller than Alpaca.

### Why this matters for your project

If you only train on translated Alpaca, the model learns "translated Urdu" — grammatically correct but stilted. Mixing in Aya teaches the model what **native** Urdu instruction-following looks like. It's a quality signal, not a volume play.

In [6]:
# Load the Aya dataset
# This is a large multilingual dataset — loading just the train split
# NOTE: This download might take a few minutes depending on your connection
print("Loading Aya dataset (this may take a minute)...")
aya = load_dataset("CohereForAI/aya_dataset", split="train")

print(f"\nTotal Aya examples (all languages): {len(aya):,}")
print(f"Columns: {aya.column_names}")
print(f"\nFirst example:")
print(aya[0])

Loading Aya dataset (this may take a minute)...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/137M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/978k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/202362 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1750 [00:00<?, ? examples/s]


Total Aya examples (all languages): 202,362
Columns: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']

First example:
{'inputs': 'Heestan waxaa qada Khalid Haref Ahmed \nOO ku Jiray Kooxdii Dur Dur!', 'targets': "Habeen ma hurdoo\nAday horjoogoo\nDharaar ma hargalo\nAduun baabay helayee\nRuntii ku helayoo\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nHannaan wanaageey\nHadal macaaneey\nWadnahaad haleeshayoo\nWaad hirgalaysaa\nRuntii ku helayoo\nMaantaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nOlolaha jacaylkeenna\nYididdiiladeeniyo\nuur midoo fiyowbaan\nku abaabulaynaa\nUbixii aan beernaan\nku intifaacsanaynaa\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nAfar gu' iyo dheeraad\nAxdigaynu taagnay\nAyaan dantiyo guur\nu adkaynay gaarnoo\nMarwadayda noqotoo\nUbad daadahaysee\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa...", 'language': 'Somali', 'language_code': 'som', 'annotation_type': 'o

### Filter to Urdu only

Aya has a `language` column (or similar). Let's find the Urdu examples and see how many we get.

In [7]:
# What languages are available? Show the top 20
from collections import Counter

# The column might be 'language' or 'language_code' — let's check
print("Columns:", aya.column_names)
print("\nSample row:")
print(aya[0])

Columns: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']

Sample row:
{'inputs': 'Heestan waxaa qada Khalid Haref Ahmed \nOO ku Jiray Kooxdii Dur Dur!', 'targets': "Habeen ma hurdoo\nAday horjoogoo\nDharaar ma hargalo\nAduun baabay helayee\nRuntii ku helayoo\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nHannaan wanaageey\nHadal macaaneey\nWadnahaad haleeshayoo\nWaad hirgalaysaa\nRuntii ku helayoo\nMaantaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nOlolaha jacaylkeenna\nYididdiiladeeniyo\nuur midoo fiyowbaan\nku abaabulaynaa\nUbixii aan beernaan\nku intifaacsanaynaa\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa\n\nAfar gu' iyo dheeraad\nAxdigaynu taagnay\nAyaan dantiyo guur\nu adkaynay gaarnoo\nMarwadayda noqotoo\nUbad daadahaysee\nCaawaan iman iman\nOonkaan u liitay\nIga ba'ay harraadkiisa...", 'language': 'Somali', 'language_code': 'som', 'annotation_type': 'original-annotations', 'user_id': 'f0ff69570af705

In [8]:
# Count examples per language (adjust column name if needed)
# Common column names in Aya: 'language', 'language_code', 'lang'
lang_col = None
for candidate in ['language', 'language_code', 'lang']:
    if candidate in aya.column_names:
        lang_col = candidate
        break

if lang_col is None:
    print("Could not find language column! Columns are:", aya.column_names)
else:
    print(f"Using language column: '{lang_col}'")
    lang_counts = Counter(aya[lang_col])
    
    # Show top 20 languages
    print(f"\nTop 20 languages (out of {len(lang_counts)} total):")
    for lang, count in lang_counts.most_common(20):
        marker = " <-- THIS IS OURS" if 'urdu' in lang.lower() or lang == 'urd' else ""
        print(f"  {lang:20s} {count:>6,}{marker}")

Using language column: 'language'

Top 20 languages (out of 71 total):
  Plateau Malagasy     14,597
  Sinhala              14,524
  Tamil                14,133
  Yoruba               11,758
  Standard Malay       10,073
  Portuguese            8,997
  Vietnamese            8,676
  Kyrgyz                8,622
  Telugu                8,439
  Moroccan Arabic       8,090
  Somali                7,704
  Panjabi               6,385
  Japanese              6,259
  Standard Arabic       4,995
  Turkish               4,046
  Nepali                4,002
  Gujarati              3,989
  English               3,944
  Spanish               3,854
  Marathi               3,545


In [9]:
# Filter to Urdu only
# The language value might be 'Urdu', 'urdu', 'urd', etc. — let's be flexible
urdu_keywords = ['urdu', 'urd']

aya_urdu = aya.filter(
    lambda ex: any(kw in ex[lang_col].lower() for kw in urdu_keywords)
)

print(f"Aya Urdu examples: {len(aya_urdu):,}")
print(f"\nThis is {100*len(aya_urdu)/len(aya):.1f}% of the full Aya dataset")

Filter:   0%|          | 0/202362 [00:00<?, ? examples/s]

Aya Urdu examples: 733

This is 0.4% of the full Aya dataset


### Look at Urdu examples

This is the important part — see what native Urdu instruction data looks like. Notice how the Urdu reads compared to what Google Translate or an LLM would produce.

In [10]:
# Show 5 Urdu examples
print(f"Columns in Aya Urdu: {aya_urdu.column_names}")
print()

for i in range(min(5, len(aya_urdu))):
    ex = aya_urdu[i]
    print(f"{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    # Print all columns to understand the structure
    for col in aya_urdu.column_names:
        val = str(ex[col])
        if len(val) > 300:
            val = val[:300] + "..."
        print(f"{col}: {val}")
    print()

Columns in Aya Urdu: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']

Example 1
inputs: نوری سال کے بارے میں کوی شعر بتایں
targets:  نوری سال کے بارے میں اشتیاق احمد کا شعر ہے:

ر و شنی سب سے تیز چلتی ہے
اس کے جیسے نہیں کسی کی چال
ایک برس میں جو سفر کرتی ہے
اس کو کہتے ہے ایک نوری سال۔
language: Urdu
language_code: urd
annotation_type: re-annotations
user_id: 634e3e0931cf859c8d8add6e8a5cc59524f2f996b15f83ee0a150ebe82324a15

Example 2
inputs: چار دوست—ایلس، باب، کیرول اور ڈیو—ایک قطار میں بیٹھے ہیں۔ ایلس کہتی ہے، "باب بہت بائیں طرف نہیں ہے۔" باب کہتے ہیں، "ڈیو بہت دائیں طرف نہیں ہے۔" کیرول کا کہنا ہے، "ایلس بہت بائیں یا بہت دائیں طرف نہیں ہے." ڈیو کہتے ہیں، "باب بائیں سے دوسرے نمبر پر ہے۔" ہر شخص کہاں بیٹھا ہے؟
targets: ایلس بہت بائیں طرف ہے، باب بائیں سے دوسرے، کیرول دائیں سے دوسرے، اور ڈیو بہت دائیں طرف ہے۔ اگر ایلس بائیں جانب نہ ہوتی تو باب کا بیان غلط ہوگا۔ اگر باب بہت بائیں طرف ہوتے تو ڈیو کا بیان غلط ہوگا۔ لہذا، ایلس بہت بائیں طرف ہے، اور ڈیو بہت دا

### Aya → Alpaca format mapping

Aya and Alpaca use different column names. Before we can combine them, we need to understand the mapping. Let's figure out which Aya columns correspond to Alpaca's `instruction`, `input`, and `output`.

In [11]:
# Show column names and types for Aya Urdu
print("Aya Urdu columns and first values:")
print("-" * 40)
for col in aya_urdu.column_names:
    val = str(aya_urdu[0][col])[:100]
    print(f"{col:25s} → {val}")

Aya Urdu columns and first values:
----------------------------------------
inputs                    → نوری سال کے بارے میں کوی شعر بتایں
targets                   →  نوری سال کے بارے میں اشتیاق احمد کا شعر ہے:

ر و شنی سب سے تیز چلتی ہے
اس کے جیسے نہیں کسی کی چال
ا
language                  → Urdu
language_code             → urd
annotation_type           → re-annotations
user_id                   → 634e3e0931cf859c8d8add6e8a5cc59524f2f996b15f83ee0a150ebe82324a15


### Urdu text length distribution

In [12]:
# Length stats for Aya Urdu
# We'll concatenate all text columns to get total length
# Adjust column names based on what you saw above

# Try common Aya column names for the text content
text_cols = [c for c in aya_urdu.column_names if c not in ['language', 'language_code', 'lang', 'dataset_name', 'id', 'annotation_type']]
print(f"Text columns (guessed): {text_cols}")

urdu_lengths = []
for ex in aya_urdu:
    total = sum(len(str(ex[c])) for c in text_cols)
    urdu_lengths.append(total)

print(f"\nAya Urdu — character length stats:")
print(f"  Min:    {min(urdu_lengths):,}")
print(f"  Max:    {max(urdu_lengths):,}")
print(f"  Mean:   {sum(urdu_lengths)/len(urdu_lengths):,.0f}")
print(f"  Median: {sorted(urdu_lengths)[len(urdu_lengths)//2]:,}")

Text columns (guessed): ['inputs', 'targets', 'user_id']

Aya Urdu — character length stats:
  Min:    81
  Max:    111,602
  Mean:   1,951
  Median: 733


---
## Part 3: Side-by-side comparison

Let's compare what you'll be working with:

In [13]:
print("DATASET COMPARISON")
print("=" * 50)
print(f"{'':25s} {'Alpaca':>10s} {'Aya Urdu':>10s}")
print("-" * 50)
print(f"{'Total examples':25s} {len(alpaca):>10,} {len(aya_urdu):>10,}")
print(f"{'Language':25s} {'English':>10s} {'Urdu':>10s}")
print(f"{'Source':25s} {'GPT-gen':>10s} {'Human':>10s}")
print(f"{'Needs translation':25s} {'YES':>10s} {'No':>10s}")
print(f"{'Avg char length':25s} {sum(lengths)//len(lengths):>10,} {sum(urdu_lengths)//len(urdu_lengths):>10,}")
print()
print("Your combined dataset target: ~50k examples")
print(f"  Translated Alpaca:     ~{len(alpaca):,} (after translation + cleaning)")
print(f"  Aya Urdu:              ~{len(aya_urdu):,} (already in Urdu)")
print(f"  Hand-crafted:          ~300-500 (you write these)")
print(f"  Estimated total:       ~{len(alpaca) + len(aya_urdu) + 400:,}")

DATASET COMPARISON
                              Alpaca   Aya Urdu
--------------------------------------------------
Total examples                51,760        733
Language                     English       Urdu
Source                       GPT-gen      Human
Needs translation                YES         No
Avg char length                  765      1,951

Your combined dataset target: ~50k examples
  Translated Alpaca:     ~51,760 (after translation + cleaning)
  Aya Urdu:              ~733 (already in Urdu)
  Hand-crafted:          ~300-500 (you write these)
  Estimated total:       ~52,893


---
## What you just learned

| Thing | Why it matters |
|-------|---------------|
| Alpaca has ~52k diverse English examples | Big volume, but needs translation — that's the expensive/slow part |
| Aya has a smaller Urdu subset | Free native Urdu, high quality — use all of it |
| Column names differ between datasets | Need a mapping step before combining |
| Length distributions | Tells you if examples will fit in 2048 token window |

## Next steps

1. **Pull and format Aya Urdu** — easiest, already in Urdu, just needs column mapping
2. **Set up Alpaca translation** — biggest effort, needs an LLM or translation API
3. **Start hand-crafting examples** — your unique contribution